In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 01 — Stratified FPG Regression Models  (REVISED, single-cell)
# ═══════════════════════════════════════════════════════════════════════════════
# Project layout assumed:
#   project_root/
#     ├── data/        (hn20–hn24 *.sas7bdat)
#     ├── notebooks/   (this notebook)
#     └── results/
#           ├── figures/
#           └── tables/
#
# Revisions relative to the submitted version (addressing referee comments):
#   [FIX-1] Sample construction reconciled: ALL FIVE KNHANES cycles (hn20–hn24)
#           are loaded, matching the Methods text (34,640 → 27,934). The submitted
#           code loaded only 3 files (→16,677) while the text described 5-file
#           numbers, producing the referee's "unexplained exclusions".
#   [FIX-2] Medication exclusion now actually runs: HE_DMdr exists only in
#           hn20/hn21, so the 3-file version silently SKIPPED the 48-participant
#           exclusion the paper claimed. With 5 files it executes correctly.
#   [FIX-3] The age≥19 step (in code, absent from the Methods text) is logged
#           explicitly as a documented CONSORT step.
#   [FIX-4] BASE-RATE DIAGNOSTICS added: majority-class baseline, improvement over
#           baseline, balanced accuracy, Cohen's κ, macro-F1, and per-group
#           confusion matrices — separating genuine predictive-equity gaps from
#           artefacts of differing subgroup class balance.
#   [FIX-5] Per-group hold-out sample sizes reported in every results table.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Standard library ──────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import logging
from pathlib import Path

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import pyreadstat
import joblib

# ── ML ────────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             balanced_accuracy_score, cohen_kappa_score,
                             f1_score, confusion_matrix)
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

SEED = 42
np.random.seed(SEED)
N_TRIALS = 25                       # Optuna trials per model (referee: report tuning)

# ═══════════════════════════════════════════════════════════════════════════════
# 0. PROJECT PATHS  (robust to where the notebook is launched)
# ═══════════════════════════════════════════════════════════════════════════════
def find_project_root(start: Path = Path.cwd()) -> Path:
    """Root = the folder that contains both 'data/' and 'results/'."""
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "results").is_dir():
            return p
    # fallback: if launched from notebooks/, go one level up
    return start.parent if start.name == "notebooks" else start

ROOT      = find_project_root()
DATA_DIR  = ROOT / "data"
FIG_DIR   = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
ART_DIR   = ROOT / "results" / "artifacts"     # intermediate .pkl / preprocessed data
for d in (FIG_DIR, TABLE_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

KNHANES_FILES = ["hn20_all.sas7bdat", "hn21_all.sas7bdat",
                 "hn22_all.sas7bdat", "hn23_all.sas7bdat",
                 "hn24_all.sas7bdat"]

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CONSTANTS & CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
def assign_glucose_stage(fpg: float) -> str:
    """Classify FPG (mg/dL) into clinical glucose stage per ADA (2023)."""
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"

STAGE_ORDER = ["normal", "ifg", "diabetes"]     # fixed label order for confusion matrices

def assign_age_group(age: float) -> float:
    """0 = Young (19–39), 1 = Middle (40–64), 2 = Elderly (65+)."""
    if age <= 39: return 0.0
    if age <= 64: return 1.0
    return 2.0

GROUP_CONFIG = {
    "Young_Male":    {"sex_code": 1.0, "age_group": 0.0},
    "Young_Female":  {"sex_code": 2.0, "age_group": 0.0},
    "Middle_Male":   {"sex_code": 1.0, "age_group": 1.0},
    "Middle_Female": {"sex_code": 2.0, "age_group": 1.0},
    "Elderly_Male":  {"sex_code": 1.0, "age_group": 2.0},
    "Elderly_Female":{"sex_code": 2.0, "age_group": 2.0},
}
GROUP_LABELS = {
    "Young_Male":"Young Male", "Young_Female":"Young Female",
    "Middle_Male":"Middle-aged Male", "Middle_Female":"Middle-aged Female",
    "Elderly_Male":"Elderly Male", "Elderly_Female":"Elderly Female",
}

ALGORITHMS     = ["LR", "Ridge", "RF", "LGBM", "XGB", "MLP"]
BASELINE_ALGOS = {"LR", "Ridge"}    # actuarial baselines — excluded from best-model pick

log.info("Constants defined: %d groups, %d algorithms", len(GROUP_CONFIG), len(ALGORITHMS))

# ═══════════════════════════════════════════════════════════════════════════════
# 2. DATA LOADING — KNHANES (5 survey cycles: 2020–2024)   [FIX-1]
# ═══════════════════════════════════════════════════════════════════════════════
KEY_COLS    = ["ID", "year", "sex", "age"]
TARGET_COLS = ["HE_glu", "HE_DMdr"]      # FPG + diabetes-medication flag
CAT_COLS = ["HE_obe","BO1_1","BO1_2","BO1_3","BD1_11","BD2_1","BS3_1",
            "BE3_71","BE3_75","BE3_81","BE3_91","pa_aerobic",
            "L_BR_FQ","BP1","mh_stress","incm","ho_incm","edu","BH1"]
NUM_COLS = ["HE_BMI","HE_wc","HE_wt","N_EN","N_CHO","N_SUGAR","N_NA",
            "N_FAT","N_SFA","N_TDF","N_K","N_PROT"]
ALL_VARS = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

frames = []
for fname in KNHANES_FILES:
    fpath = DATA_DIR / fname
    if not fpath.exists():
        raise FileNotFoundError(
            f"Missing KNHANES file: {fname}\n"
            f"Place all five cycles (hn20–hn24) in the project's data/ folder "
            f"(download from https://knhanes.kdca.go.kr).")
    df_yr, _ = pyreadstat.read_sas7bdat(str(fpath))
    keep = [v for v in ALL_VARS if v in df_yr.columns]
    frames.append(df_yr[keep].copy())
    log.info("Loaded %-16s : %6d rows | HE_DMdr present = %s",
             fname, len(df_yr), "HE_DMdr" in df_yr.columns)

df_raw = pd.concat(frames, axis=0, ignore_index=True)
log.info("Merged dataset: %d rows × %d columns", *df_raw.shape)

# ═══════════════════════════════════════════════════════════════════════════════
# 3. PREPROCESSING — explicit, logged CONSORT flow   [FIX-2, FIX-3]
# ═══════════════════════════════════════════════════════════════════════════════
consort = []
n0 = len(df_raw)
consort.append(("Raw merged (hn20–hn24)", None, n0, None))

# Step 1 — drop missing FPG
df = df_raw.dropna(subset=["HE_glu"]).reset_index(drop=True)
consort.append(("1. Drop missing FPG (HE_glu)", n0, len(df), n0 - len(df)))
log.info("Step 1 — Drop missing FPG:      %6d → %6d (removed %d)", n0, len(df), n0 - len(df))

# Step 2 — exclude diabetes-medication users (HE_DMdr == 1)
n1 = len(df)
if "HE_DMdr" in df.columns:
    # HE_DMdr present only in some cycles; NaN (column absent that cycle) is
    # treated as "not medicated" and retained. Only explicit ==1 is excluded.
    df = df[df["HE_DMdr"] != 1].reset_index(drop=True)
    consort.append(("2. Exclude diabetes-medication users", n1, len(df), n1 - len(df)))
    log.info("Step 2 — Exclude medicated:     %6d → %6d (removed %d)", n1, len(df), n1 - len(df))
else:
    consort.append(("2. Exclude medicated (col absent)", n1, len(df), 0))
    log.warning("Step 2 — HE_DMdr absent entirely; exclusion skipped.")

# Step 3 — restrict to adults (age ≥ 19)   [previously undocumented in text]
n2 = len(df)
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df[df["age"] >= 19].reset_index(drop=True)
consort.append(("3. Restrict to adults (age ≥ 19)", n2, len(df), n2 - len(df)))
log.info("Step 3 — Age ≥ 19:              %6d → %6d (removed %d)", n2, len(df), n2 - len(df))

# Step 4 — physiological FPG bounds (40–400 mg/dL)
n3 = len(df)
df = df[(df["HE_glu"] >= 40) & (df["HE_glu"] <= 400)].reset_index(drop=True)
consort.append(("4. FPG within 40–400 mg/dL", n3, len(df), n3 - len(df)))
log.info("Step 4 — FPG outlier removal:   %6d → %6d (removed %d)", n3, len(df), n3 - len(df))

log.info(">>> Final analytical sample: %d participants", len(df))

# Derived columns
df["age_group"]    = df["age"].apply(assign_age_group)
df["GlucoseStage"] = df["HE_glu"].apply(assign_glucose_stage)

# Rename to interpretable English feature names
COLUMN_MAP = {
    "ID":"ID","year":"SurveyYear","sex":"Sex","age":"Age","age_group":"AgeGroup",
    "HE_glu":"FPG","HE_obe":"ObesityStatus","BO1_1":"WeightChangeStatus",
    "BO1_2":"WeightLossAmount","BO1_3":"WeightGainAmount","BD1_11":"DrinkingFrequency",
    "BD2_1":"DrinkingAmount","BS3_1":"SmokingStatus","BE3_71":"VigorousAct_Work",
    "BE3_75":"VigorousAct_Leisure","BE3_81":"ModerateAct_Work","BE3_91":"WalkingActivity",
    "pa_aerobic":"AerobicRate","L_BR_FQ":"BreakfastFreq","BP1":"StressLevel",
    "mh_stress":"StressAwareness","incm":"IncomeQuartile","ho_incm":"HouseholdIncome",
    "edu":"EducationLevel","BH1":"HealthScreening","HE_BMI":"BMI","HE_wc":"WaistCirc",
    "HE_wt":"Weight","N_EN":"Energy_kcal","N_CHO":"Carb_g","N_SUGAR":"Sugar_g",
    "N_NA":"Sodium_mg","N_FAT":"Fat_g","N_SFA":"SatFat_g","N_TDF":"Fiber_g",
    "N_K":"Potassium_mg","N_PROT":"Protein_g",
}
df.rename(columns=COLUMN_MAP, inplace=True)

# Reproducible CONSORT table
consort_df = pd.DataFrame(consort, columns=["Step", "n_before", "n_after", "removed"])
print("\n── CONSORT sample-construction flow ─────────────────────────────")
print(consort_df.to_string(index=False))
consort_df.to_csv(TABLE_DIR / "consort_flow.csv", index=False, encoding="utf-8")

df.to_csv(ART_DIR / "knhanes_fpg_preprocessed.csv", index=False, encoding="utf-8")
log.info("Preprocessed dataset saved: %d rows × %d cols", *df.shape)

# ═══════════════════════════════════════════════════════════════════════════════
# 4. FEATURE DEFINITION & FINAL DATASET
# ═══════════════════════════════════════════════════════════════════════════════
CAT_FEATURES = ["ObesityStatus","WeightChangeStatus","WeightLossAmount","WeightGainAmount",
                "DrinkingFrequency","DrinkingAmount","SmokingStatus","VigorousAct_Work",
                "VigorousAct_Leisure","ModerateAct_Work","WalkingActivity","AerobicRate",
                "BreakfastFreq","StressLevel","StressAwareness","IncomeQuartile",
                "HouseholdIncome","EducationLevel","HealthScreening"]
NUM_FEATURES = ["BMI","WaistCirc","Weight","Energy_kcal","Carb_g","Sugar_g","Sodium_mg",
                "Fat_g","SatFat_g","Fiber_g","Potassium_mg","Protein_g"]
X_FEATURES = NUM_FEATURES + CAT_FEATURES

required = X_FEATURES + ["FPG","GlucoseStage","Sex","AgeGroup","SurveyYear"]
df_final = df[[c for c in required if c in df.columns]].copy()
for col in df_final.columns:
    if col != "GlucoseStage":
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce").fillna(0)
log.info("df_final shape: %s | %d predictors", df_final.shape, len(X_FEATURES))

# ═══════════════════════════════════════════════════════════════════════════════
# 5. TABLE 1 — DESCRIPTIVE STATISTICS + BASE RATES BY GROUP   [FIX-5]
# ═══════════════════════════════════════════════════════════════════════════════
rows = []
for grp, cfg in GROUP_CONFIG.items():
    sub = df_final[(df_final["AgeGroup"] == cfg["age_group"]) &
                   (df_final["Sex"] == cfg["sex_code"])]
    vc = sub["GlucoseStage"].value_counts(normalize=True) * 100
    maj_stage = sub["GlucoseStage"].value_counts().idxmax()
    rows.append({
        "Group":GROUP_LABELS[grp], "n":len(sub),
        "FPG_Mean":round(sub["FPG"].mean(),2), "FPG_SD":round(sub["FPG"].std(),2),
        "FPG_Median":round(sub["FPG"].median(),1),
        "Normal_%":round(vc.get("normal",0),1), "IFG_%":round(vc.get("ifg",0),1),
        "Diabetes_%":round(vc.get("diabetes",0),1),
        "Majority_stage":maj_stage, "Majority_baseline_%":round(vc.max(),1),
    })
table1 = pd.DataFrame(rows)
tot = {"Group":"Total","n":len(df_final),
       "FPG_Mean":round(df_final["FPG"].mean(),2),"FPG_SD":round(df_final["FPG"].std(),2),
       "FPG_Median":round(df_final["FPG"].median(),1)}
vc_all = df_final["GlucoseStage"].value_counts(normalize=True)*100
tot.update({"Normal_%":round(vc_all.get("normal",0),1),"IFG_%":round(vc_all.get("ifg",0),1),
            "Diabetes_%":round(vc_all.get("diabetes",0),1),
            "Majority_stage":df_final["GlucoseStage"].value_counts().idxmax(),
            "Majority_baseline_%":round(vc_all.max(),1)})
table1 = pd.concat([table1, pd.DataFrame([tot])], ignore_index=True)
print("\n── Table 1: Descriptive statistics + base rates ─────────────────")
print(table1.to_string(index=False))
table1.to_csv(TABLE_DIR / "table1_descriptive_stats.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. MODEL TRAINING FUNCTIONS (Optuna-tuned)
# ═══════════════════════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_pred) -> dict:
    return {"RMSE":round(float(np.sqrt(mean_squared_error(y_true,y_pred))),4),
            "MAE": round(float(mean_absolute_error(y_true,y_pred)),4),
            "R2":  round(float(r2_score(y_true,y_pred)),4)}

def train_model(X_train, X_val, y_train, y_val, algorithm, n_trials=N_TRIALS):
    """Train one algorithm with Optuna TPE tuning. Returns (model, metrics, params)."""
    def mk_study():
        return optuna.create_study(direction="maximize",
                                   sampler=optuna.samplers.TPESampler(seed=SEED))
    def score(model):
        model.fit(X_train, y_train)
        return -mean_squared_error(y_val, model.predict(X_val))

    best_params = {}
    if algorithm == "LR":
        model = LinearRegression().fit(X_train, y_train)

    elif algorithm == "Ridge":
        s = mk_study()
        s.optimize(lambda t: score(Ridge(
            alpha=t.suggest_float("alpha",1e-3,100.0,log=True), random_state=SEED)),
            n_trials=n_trials)
        best_params = s.best_params
        model = Ridge(**best_params, random_state=SEED).fit(X_train, y_train)

    elif algorithm == "RF":
        s = mk_study()
        s.optimize(lambda t: score(RandomForestRegressor(
            n_estimators=t.suggest_int("n_estimators",100,500),
            max_depth=t.suggest_int("max_depth",3,15),
            min_samples_split=t.suggest_int("min_samples_split",2,10),
            random_state=SEED, n_jobs=-1)), n_trials=n_trials)
        best_params = s.best_params
        model = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=-1).fit(X_train,y_train)

    elif algorithm == "LGBM":
        s = mk_study()
        s.optimize(lambda t: score(lgb.LGBMRegressor(
            n_estimators=t.suggest_int("n_estimators",100,500),
            max_depth=t.suggest_int("max_depth",3,7),
            learning_rate=t.suggest_float("learning_rate",0.01,0.1,log=True),
            subsample=t.suggest_float("subsample",0.7,1.0),
            colsample_bytree=t.suggest_float("colsample_bytree",0.7,1.0),
            random_state=SEED, n_jobs=-1, verbose=-1)), n_trials=n_trials)
        best_params = s.best_params
        model = lgb.LGBMRegressor(**best_params, random_state=SEED, n_jobs=-1, verbose=-1).fit(X_train,y_train)

    elif algorithm == "XGB":
        s = mk_study()
        s.optimize(lambda t: score(xgb.XGBRegressor(
            n_estimators=t.suggest_int("n_estimators",100,500),
            max_depth=t.suggest_int("max_depth",3,7),
            learning_rate=t.suggest_float("learning_rate",0.01,0.1,log=True),
            subsample=t.suggest_float("subsample",0.7,1.0),
            colsample_bytree=t.suggest_float("colsample_bytree",0.7,1.0),
            random_state=SEED, tree_method="hist", verbosity=0)), n_trials=n_trials)
        best_params = s.best_params
        model = xgb.XGBRegressor(**best_params, random_state=SEED, tree_method="hist", verbosity=0).fit(X_train,y_train)

    elif algorithm == "MLP":
        s = mk_study()
        def _mlp(t):
            return MLPRegressor(
                hidden_layer_sizes=tuple([t.suggest_int("n_units",32,256)]*t.suggest_int("n_layers",1,3)),
                alpha=t.suggest_float("alpha",1e-5,1e-2,log=True),
                learning_rate_init=t.suggest_float("lr_init",1e-4,1e-2,log=True),
                max_iter=300, random_state=SEED)
        s.optimize(lambda t: score(_mlp(t)), n_trials=n_trials)
        bp = s.best_params; best_params = bp
        model = MLPRegressor(hidden_layer_sizes=tuple([bp["n_units"]]*bp["n_layers"]),
                             alpha=bp["alpha"], learning_rate_init=bp["lr_init"],
                             max_iter=300, random_state=SEED).fit(X_train,y_train)
    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")

    return model, compute_metrics(y_val, model.predict(X_val)), best_params

log.info("Model training functions defined (Optuna n_trials=%d).", N_TRIALS)

# ═══════════════════════════════════════════════════════════════════════════════
# 7. STRATIFIED TRAINING — 6 groups × 6 algorithms
# ═══════════════════════════════════════════════════════════════════════════════
all_results       = {}    # {group: {algo: metrics}}
best_models       = {}    # {group: fitted best model}
best_algo_name    = {}    # {group: algo str}
best_params_store = {}     # {(group, algo): params}
holdout_index     = {}     # {group: [X_val.index]} — reproducible downstream eval

for grp, cfg in GROUP_CONFIG.items():
    df_g = df_final[(df_final["AgeGroup"] == cfg["age_group"]) &
                    (df_final["Sex"] == cfg["sex_code"])].copy()
    log.info("="*60)
    log.info("Group: %-18s (n=%d)", GROUP_LABELS[grp], len(df_g))

    X, y = df_g[X_FEATURES], df_g["FPG"]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED)
    holdout_index[grp] = X_val.index.tolist()

    all_results[grp] = {}
    best_rmse, best_model, best_algo = float("inf"), None, None
    for algo in ALGORITHMS:
        try:
            model, metrics, bp = train_model(X_train, X_val, y_train, y_val, algo)
            all_results[grp][algo] = metrics
            best_params_store[(grp, algo)] = bp
            tag = " [baseline]" if algo in BASELINE_ALGOS else ""
            log.info("  %-6s | RMSE=%.3f MAE=%.3f R²=%.3f%s",
                     algo, metrics["RMSE"], metrics["MAE"], metrics["R2"], tag)
            if (algo not in BASELINE_ALGOS) and (metrics["RMSE"] < best_rmse):
                best_rmse, best_model, best_algo = metrics["RMSE"], model, algo
        except Exception as exc:
            log.warning("  %-6s | FAILED: %s", algo, exc)
            all_results[grp][algo] = None

    best_models[grp]    = best_model
    best_algo_name[grp] = best_algo
    log.info("  >>> Best (non-baseline): %s (RMSE=%.3f)", best_algo, best_rmse)

log.info("Stratified training complete.")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. TABLE 2 + BASE-RATE-CORRECTED FAIRNESS DIAGNOSTICS   [FIX-4]  ★ core revision
# ═══════════════════════════════════════════════════════════════════════════════
# Raw Stage Accuracy is confounded by subgroup class balance (a 90%-Normal group
# scores ~90% by always predicting "Normal"). We therefore add:
#   • Majority baseline — accuracy of the trivial "always predict majority" rule
#   • Improvement       — StageAcc − MajorityBaseline (skill beyond base rate)
#   • Balanced accuracy — mean per-class recall (base-rate invariant)
#   • Cohen's κ         — agreement corrected for chance
#   • Macro-F1          — class-size-invariant F1
# plus per-group confusion matrices.
# ───────────────────────────────────────────────────────────────────────────────
def stage_of(vals):
    return pd.Series([assign_glucose_stage(v) for v in np.asarray(vals)], dtype=object)

diag_rows, confusions = [], {}
for grp, cfg in GROUP_CONFIG.items():
    model = best_models.get(grp)
    if model is None:
        continue
    df_g = df_final[(df_final["AgeGroup"] == cfg["age_group"]) &
                    (df_final["Sex"] == cfg["sex_code"])].copy()
    X, y = df_g[X_FEATURES], df_g["FPG"]
    _, X_val, _, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED)

    y_pred = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))
    mae  = float(mean_absolute_error(y_val, y_pred))
    r2   = float(r2_score(y_val, y_pred))
    bias = float(np.mean(y_pred - y_val.values))

    st, sp = stage_of(y_val.values), stage_of(y_pred)
    stage_acc = float((st.values == sp.values).mean())
    maj_cls   = st.value_counts().idxmax()
    maj_base  = float((st.values == maj_cls).mean())
    bal_acc   = float(balanced_accuracy_score(st, sp))
    kappa     = float(cohen_kappa_score(st, sp, labels=STAGE_ORDER))
    macro_f1  = float(f1_score(st, sp, average="macro", labels=STAGE_ORDER))
    confusions[grp] = confusion_matrix(st, sp, labels=STAGE_ORDER)

    diag_rows.append({
        "Group":GROUP_LABELS[grp], "Algorithm":best_algo_name[grp], "n_test":int(len(y_val)),
        "RMSE":round(rmse,2), "MAE":round(mae,2), "R2":round(r2,3), "Bias":round(bias,2),
        "StageAcc":round(stage_acc,3), "MajBaseline":round(maj_base,3),
        "Improvement":round(stage_acc-maj_base,3),          # ★ skill over base rate
        "BalancedAcc":round(bal_acc,3), "CohenKappa":round(kappa,3), "MacroF1":round(macro_f1,3),
    })

diag = pd.DataFrame(diag_rows)

def gap(col): return round(diag[col].max() - diag[col].min(), 3)
disparity = pd.DataFrame([{
    "ΔStageAcc (raw, confounded)":        gap("StageAcc"),
    "ΔImprovement (over majority)":       gap("Improvement"),
    "ΔBalancedAcc (base-rate invariant)": gap("BalancedAcc"),
    "ΔCohenKappa":                        gap("CohenKappa"),
    "ΔMacroF1":                           gap("MacroF1"),
    "ΔRMSE":                              gap("RMSE"),
}]).T.rename(columns={0:"Value"})

print("\n── Table 2 (revised): Hold-out performance + base-rate diagnostics ──")
print(diag.to_string(index=False))
print("\n── Cross-group disparities: raw vs base-rate-corrected metrics ──")
print(disparity.to_string())

print("\n── Per-group confusion matrices (rows=true, cols=pred; order: normal, ifg, diabetes) ──")
for grp in GROUP_CONFIG:
    if grp in confusions:
        n_test = diag.loc[diag["Group"]==GROUP_LABELS[grp],"n_test"].values[0]
        print(f"\n{GROUP_LABELS[grp]} (n_test={n_test}):")
        print(pd.DataFrame(confusions[grp], index=STAGE_ORDER, columns=STAGE_ORDER).to_string())

diag.to_csv(TABLE_DIR / "table2_performance_diagnostics.csv", index=False, encoding="utf-8")
disparity.to_csv(TABLE_DIR / "table2_disparities.csv", encoding="utf-8")
joblib.dump(confusions, ART_DIR / "confusion_matrices.pkl")

# ═══════════════════════════════════════════════════════════════════════════════
# 9. SAVE ARTEFACTS  (→ Notebooks 02–04)
# ═══════════════════════════════════════════════════════════════════════════════
try:
    df_final.to_parquet(ART_DIR / "df_final.parquet", index=False)
except (ImportError, ValueError):
    df_final.to_csv(ART_DIR / "df_final.csv", index=False, encoding="utf-8")
    log.warning("parquet engine unavailable; df_final saved as CSV instead.")

joblib.dump(all_results,       ART_DIR / "all_results.pkl")
joblib.dump(best_models,       ART_DIR / "best_models.pkl")
joblib.dump(best_algo_name,    ART_DIR / "best_algo_name.pkl")
joblib.dump(best_params_store, ART_DIR / "best_params_store.pkl")
joblib.dump(holdout_index,     ART_DIR / "holdout_index.pkl")
joblib.dump(diag,              ART_DIR / "fairness_diagnostics.pkl")
joblib.dump({"X_FEATURES":X_FEATURES,"NUM_FEATURES":NUM_FEATURES,
             "CAT_FEATURES":CAT_FEATURES,"GROUP_CONFIG":GROUP_CONFIG,
             "GROUP_LABELS":GROUP_LABELS,"STAGE_ORDER":STAGE_ORDER,"SEED":SEED},
            ART_DIR / "config.pkl")

log.info("All artefacts saved to results/artifacts/")
log.info("Notebook 01 complete.")

2026-09-01 13:50:46,114 | INFO | Constants defined: 6 groups, 6 algorithms
2026-09-01 13:50:47,438 | INFO | Loaded hn20_all.sas7bdat :   7359 rows | HE_DMdr present = True
2026-09-01 13:50:48,729 | INFO | Loaded hn21_all.sas7bdat :   7090 rows | HE_DMdr present = True
2026-09-01 13:50:49,540 | INFO | Loaded hn22_all.sas7bdat :   6265 rows | HE_DMdr present = False
2026-09-01 13:50:50,425 | INFO | Loaded hn23_all.sas7bdat :   6929 rows | HE_DMdr present = False
2026-09-01 13:50:51,525 | INFO | Loaded hn24_all.sas7bdat :   6997 rows | HE_DMdr present = False
2026-09-01 13:50:51,536 | INFO | Merged dataset: 34640 rows × 37 columns
2026-09-01 13:50:51,568 | INFO | Step 1 — Drop missing FPG:       34640 →  30392 (removed 4248)
2026-09-01 13:50:51,582 | INFO | Step 2 — Exclude medicated:      30392 →  30344 (removed 48)
2026-09-01 13:50:51,605 | INFO | Step 3 — Age ≥ 19:               30344 →  27936 (removed 2408)
2026-09-01 13:50:51,620 | INFO | Step 4 — FPG outlier removal:    27936 →  279


── CONSORT sample-construction flow ─────────────────────────────
                                Step  n_before  n_after  removed
              Raw merged (hn20–hn24)       NaN    34640      NaN
        1. Drop missing FPG (HE_glu)   34640.0    30392   4248.0
2. Exclude diabetes-medication users   30392.0    30344     48.0
    3. Restrict to adults (age ≥ 19)   30344.0    27936   2408.0
          4. FPG within 40–400 mg/dL   27936.0    27934      2.0


2026-09-01 13:50:52,669 | INFO | Preprocessed dataset saved: 27934 rows × 39 cols
2026-09-01 13:50:52,720 | INFO | df_final shape: (27934, 36) | 31 predictors
2026-09-01 13:50:52,815 | INFO | Model training functions defined (Optuna n_trials=25).
2026-09-01 13:50:52,825 | INFO | ============================================================
2026-09-01 13:50:52,825 | INFO | Group: Young Male         (n=3065)
2026-09-01 13:50:52,856 | INFO |   LR     | RMSE=11.377 MAE=7.167 R²=0.069 [baseline]



── Table 1: Descriptive statistics + base rates ─────────────────
             Group     n  FPG_Mean  FPG_SD  FPG_Median  Normal_%  IFG_%  Diabetes_% Majority_stage  Majority_baseline_%
        Young Male  3065     94.79   16.34        93.0      79.3   18.6         2.2         normal                 79.3
      Young Female  3594     90.85   13.84        90.0      89.8    9.1         1.1         normal                 89.8
  Middle-aged Male  5477    107.42   26.32       100.0      47.0   39.9        13.0         normal                 47.0
Middle-aged Female  7302     99.53   20.68        95.0      66.3   27.7         6.1         normal                 66.3
      Elderly Male  3692    109.51   25.19       103.0      40.8   42.1        17.0            ifg                 42.1
    Elderly Female  4804    105.69   22.91       100.0      49.9   37.6        12.5         normal                 49.9
             Total 27934    101.82   22.68        96.0      60.8   30.3         8.9         n

2026-09-01 13:50:53,065 | INFO |   Ridge  | RMSE=11.376 MAE=7.162 R²=0.069 [baseline]
2026-09-01 13:51:47,530 | INFO |   RF     | RMSE=11.421 MAE=7.187 R²=0.062
2026-09-01 13:51:58,743 | INFO |   LGBM   | RMSE=11.239 MAE=7.017 R²=0.091
2026-09-01 13:52:10,215 | INFO |   XGB    | RMSE=11.245 MAE=7.064 R²=0.090
2026-09-01 13:54:47,686 | INFO |   MLP    | RMSE=13.414 MAE=9.151 R²=-0.294
2026-09-01 13:54:47,686 | INFO |   >>> Best (non-baseline): LGBM (RMSE=11.239)
2026-09-01 13:54:47,692 | INFO | ============================================================
2026-09-01 13:54:47,697 | INFO | Group: Young Female       (n=3594)
2026-09-01 13:54:47,714 | INFO |   LR     | RMSE=14.921 MAE=7.134 R²=0.095 [baseline]
2026-09-01 13:54:47,941 | INFO |   Ridge  | RMSE=14.915 MAE=7.113 R²=0.095 [baseline]
2026-09-01 13:56:27,766 | INFO |   RF     | RMSE=14.485 MAE=7.261 R²=0.147
2026-09-01 13:56:41,736 | INFO |   LGBM   | RMSE=14.501 MAE=7.131 R²=0.145
2026-09-01 13:56:56,496 | INFO |   XGB    | RMSE=1


── Table 2 (revised): Hold-out performance + base-rate diagnostics ──
             Group Algorithm  n_test  RMSE   MAE    R2  Bias  StageAcc  MajBaseline  Improvement  BalancedAcc  CohenKappa  MacroF1
        Young Male      LGBM     613 11.24  7.02 0.091  0.92     0.790        0.783        0.007        0.407       0.259    0.415
      Young Female       XGB     719 14.38  7.16 0.159 -0.53     0.880        0.890       -0.010        0.402       0.256    0.411
  Middle-aged Male       XGB    1096 25.93 15.33 0.061 -0.25     0.462        0.483       -0.021        0.380       0.102    0.307
Middle-aged Female       XGB    1461 21.19 10.98 0.119  0.00     0.625        0.656       -0.031        0.437       0.248    0.424
      Elderly Male        RF     739 26.30 16.59 0.019 -0.49     0.441        0.438        0.003        0.341       0.008    0.220
    Elderly Female       XGB     961 24.42 15.53 0.043 -0.95     0.417        0.504       -0.086        0.362       0.058    0.280

── Cross-gr

2026-09-01 14:23:53,081 | INFO | All artefacts saved to results/artifacts/
2026-09-01 14:23:53,083 | INFO | Notebook 01 complete.
